In [ ]:

# to install package
#pip install matplotlib
#pip install pandas 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
%matplotlib inline
# Read data file and turn yes and no into 1's and 0's 
df = pd.read_csv('bankruptcy.csv')
if df['Bankrupt'].dtype == object:
    df['Bankrupt'] = (df['Bankrupt'].astype(str).str.lower() == 'yes').astype(int)
print('Shape:', df.shape)
df.head()
df['Bankrupt.head'].head()
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes.value_counts())
print(Bankrupt)
df.describe(include='all').T.head(20)

df.describe(include='all').T.head(20)

print('Duplicate rows:', df.duplicated().sum())
print('Memory (MB):', round(df.memory_usage(deep=True).sum() / 1024**2, 2))

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)
if len(missing_df):
    print(missing_df)
else:
    print('No explicit NaN values')


    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts = df['Bankrupt'].value_counts().sort_index()
sns.countplot(x='Bankrupt', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Target Class Counts')
axes[0].set_xticklabels(['Healthy', 'Bankrupt'])
axes[1].pie(target_counts, labels=['Healthy', 'Bankrupt'], autopct='%1.1f%%',
            colors=['#66c2a5', '#fc8d62'], startangle=90)
axes[1].set_title('Class Proportion')
plt.tight_layout(); plt.show()
print('\nClass balance (%):')
print((target_counts / target_counts.sum() * 100).round(2))


numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Bankrupt' in numeric_cols:
    numeric_cols.remove('Bankrupt')
print(f'Numeric features ({len(numeric_cols)}):')
print(numeric_cols)


import math
n_show = min(len(numeric_cols), 12)
ncols = 3
nrows = math.ceil(n_show / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.3*nrows))
axes = np.array(axes).reshape(-1)
for i, col in enumerate(numeric_cols[:n_show]):
    df[col].hist(ax=axes[i], bins=30, color='steelblue', edgecolor='black')
    axes[i].set_title(col)
for j in range(n_show, len(axes)):
    axes[j].axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.3*nrows))
axes = np.array(axes).reshape(-1)
for i, col in enumerate(numeric_cols[:n_show]):
    sns.boxplot(x=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col)
for j in range(n_show, len(axes)):
    axes[j].axis('off')
plt.tight_layout(); plt.show()

corrs_all = df[numeric_cols + ['Bankrupt']].corr()['Bankrupt'].abs().sort_values(ascending=False)
top_features = [c for c in corrs_all.index if c != 'Bankrupt'][:6]
print('Top features by |correlation| with target:')
print(corrs_all[top_features].round(3))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
labels = ['Healthy', 'Bankrupt']
colors = ['steelblue', 'tomato']
for i, col in enumerate(top_features[:6]):
    for cls, lbl, c in [(0, labels[0], colors[0]), (1, labels[1], colors[1])]:
        axes[i].hist(df[df['Bankrupt']==cls][col].dropna(), bins=25, alpha=0.6, label=lbl, color=c)
    axes[i].set_title(col); axes[i].legend()
plt.tight_layout(); plt.show()

means = df.groupby('Bankrupt')[numeric_cols].mean().T
means.columns = ['Healthy', 'Bankrupt']
means['Difference'] = means.iloc[:, 1] - means.iloc[:, 0]
means.sort_values('Difference', key=abs, ascending=False).round(3).head(15)

# Restrict to top numeric features for legibility
top_numeric = corrs_all.head(15).index.tolist()
if 'Bankrupt' not in top_numeric:
    top_numeric.append('Bankrupt')
corr_matrix = df[top_numeric].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.7})
plt.title('Top features — correlation matrix'); plt.tight_layout(); plt.show()

target_corr = df[numeric_cols + ['Bankrupt']].corr()['Bankrupt'].drop('Bankrupt').sort_values()
plt.figure(figsize=(10, max(5, len(target_corr)*0.25)))
colors = ['tomato' if v > 0 else 'steelblue' for v in target_corr]
target_corr.plot(kind='barh', color=colors)
plt.title('Correlation with target')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout(); plt.show()


# Top 8 features most correlated with bankruptcy
corrs = df.corr()['Bankrupt'].drop('Bankrupt').sort_values(key=abs, ascending=False).head(8)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(corrs.index):
    sns.boxplot(x='Bankrupt', y=col, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(col[:40], fontsize=9)
plt.tight_layout(); plt.show()

# Class imbalance summary
n_total = len(df); n_bk = df['Bankrupt'].sum()
print(f'Healthy : {n_total - n_bk:>5} ({(n_total - n_bk)/n_total:.2%})')
print(f'Bankrupt: {n_bk:>5} ({n_bk/n_total:.2%})')
print(f'Imbalance ratio (healthy:bankrupt): {(n_total - n_bk) / n_bk:.1f}:1')

summary = pd.DataFrame({
    'Metric': ['Total samples', 'Total features', 'Class 0 count', 'Class 1 count',
               'Class imbalance ratio', 'Top correlated feature', 'Top correlation value'],
    'Value': [
        len(df),
        df.shape[1] - 1,
        int((df['Bankrupt'] == 0).sum()),
        int((df['Bankrupt'] == 1).sum()),
        f"{(df['Bankrupt'] == 0).sum() / max(1, (df['Bankrupt'] == 1).sum()):.2f}:1",
        top_features[0],
        round(corrs_all[top_features[0]], 3),
    ],
})
summary


Shape: (6819, 95)


KeyError: 'Bankrupt.head'

In [39]:
print(df['Bankrupt'])

0        No
1        No
2        No
3        No
4        No
       ... 
6814     No
6815     No
6816     No
6817    Yes
6818     No
Name: Bankrupt, Length: 6819, dtype: str
